In [1]:
import pandas as pd
from deep_translator import GoogleTranslator

from pathlib import Path
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import pandas as pd

# from sentence_transformers import SentenceTransformer
# from sklearn.ensemble import GradientBoostingClassifier
# from sklearn.metrics import classification_report
# from sklearn.model_selection import train_test_split
# import joblib

START_YEAR = 2019
# embedder = SentenceTransformer('all-MiniLM-L6-v2')

In [2]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

In [3]:
def embed(texts, batch_size=32):
    embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, return_tensors='pt').to(device)
            output = model(**encoded)
            cls_embeddings = output.last_hidden_state[:, 0, :]  # [CLS] token
            embeddings.append(cls_embeddings.cpu())
    return torch.cat(embeddings)

CIHR

In [4]:
cihr_path = "raw_data/CIHR/"
cihr_files = Path(cihr_path).glob("*.csv")

CIHR_DFS = [pd.read_csv(f) for f in cihr_files]
CIHR_DATA = pd.concat(CIHR_DFS, ignore_index=True)

In [5]:
grant_descriptors = [
    "ApplicationTitle_TitreDemande", "PrimaryThemeEN_ThemePrincipalAN", "AllResearchCategoriesEN_TousCategoriesRechercheAN", "ApplicationKeywords_MotsClesDemande"
]

col_names = [
    'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]

CIHR_DATA = CIHR_DATA[grant_descriptors]
CIHR_DATA.columns = col_names

CIHR_DATA.drop_duplicates(inplace=True)
CIHR_DATA["Main_Discipline"].value_counts()

Main_Discipline
Biomedical                                         8989
Clinical                                           3869
Social/Cultural/Environmental/Population Health    3003
Health systems/services                            2830
Not applicable/Specified                            127
Name: count, dtype: int64

Training CIHR Main Discipline

In [6]:
tmp_data = CIHR_DATA.sample(frac=1).reset_index(drop=True) # shuffle

# tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data[tmp_data["Main_Discipline"] != "Not applicable/Specified"]

tmp_data = tmp_data.dropna(subset=['Main_Discipline'])
tmp_data["Main_Discipline"].value_counts()


Main_Discipline
Biomedical                                         8989
Clinical                                           3869
Social/Cultural/Environmental/Population Health    3003
Health systems/services                            2830
Name: count, dtype: int64

In [7]:
class CIHR_MD_Model(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [8]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = CIHR_MD_Model(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [9]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 252.6625
Epoch 2: Loss = 182.7293
Epoch 3: Loss = 165.9550
Epoch 4: Loss = 159.0168
Epoch 5: Loss = 155.7317
Epoch 6: Loss = 153.4208
Epoch 7: Loss = 151.9791
Epoch 8: Loss = 149.8181
Epoch 9: Loss = 148.5702
Epoch 10: Loss = 147.4514
Epoch 11: Loss = 146.2805
Epoch 12: Loss = 145.2543
Epoch 13: Loss = 144.0905
Epoch 14: Loss = 143.2702
Epoch 15: Loss = 142.3759
Epoch 16: Loss = 141.7622
Epoch 17: Loss = 140.7881
Epoch 18: Loss = 139.2441
Epoch 19: Loss = 139.2842
Epoch 20: Loss = 138.3915
Epoch 21: Loss = 137.3716
Epoch 22: Loss = 136.4290
Epoch 23: Loss = 135.8248
Epoch 24: Loss = 135.7129
Epoch 25: Loss = 134.1647
Epoch 26: Loss = 133.9465
Epoch 27: Loss = 132.6800
Epoch 28: Loss = 132.4404
Epoch 29: Loss = 130.9158
Epoch 30: Loss = 130.6433
Epoch 31: Loss = 130.0562
Epoch 32: Loss = 129.2830
Epoch 33: Loss = 128.6230
Epoch 34: Loss = 128.0621
Epoch 35: Loss = 127.3648
Epoch 36: Loss = 126.0871
Epoch 37: Loss = 125.6162
Epoch 38: Loss = 124.5525
Epoch 39: Loss = 124.

In [10]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()

print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                 precision    recall  f1-score   support

                                     Biomedical       0.89      0.89      0.89      1798
                                       Clinical       0.59      0.59      0.59       774
                        Health systems/services       0.60      0.59      0.60       566
Social/Cultural/Environmental/Population Health       0.66      0.66      0.66       601

                                       accuracy                           0.74      3739
                                      macro avg       0.68      0.68      0.68      3739
                                   weighted avg       0.74      0.74      0.74      3739



In [11]:
torch.save(clf_model.state_dict(), "models/CIHR_MD.pt")

In [12]:
# tmp_data = CIHR_DATA.copy()

# tmp_data["Area_of_Research"].value_counts()

NSERC

In [13]:
nserc_path = "raw_data/NSERC/"
nserc_files = Path(nserc_path).glob("*.csv")

NSERC_DFS = [pd.read_csv(f) for f in nserc_files]
NSERC_DATA = pd.concat(NSERC_DFS, ignore_index=True)

In [14]:
grant_descriptors = [
    "ApplicationTitle", "AreaOfApplicationGroupEN", "ResearchSubjectEN", "Keyword"
]

col_names = [
     'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]

NSERC_DATA = NSERC_DATA[grant_descriptors]
NSERC_DATA.columns = col_names

NSERC_DATA.drop_duplicates(inplace=True)

Model for Main Discipline

In [15]:
tmp_data = NSERC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

tmp_data = tmp_data[tmp_data["Main_Discipline"] != "Not available"]

tmp_data["Main_Discipline"].value_counts()

Main_Discipline
Advancement of knowledge                   20177
Manufacturing processes and products        4626
Information and communication services      3936
Environment                                 3927
Energy resources                            3123
Health, education and social services       2937
Transportation systems and services         2077
Agriculture and primary food production     1541
Construction, urban and rural planning      1376
Northern development                        1203
Natural resources (economic aspects)        1106
The socioeconomic objective available        737
Commercial services                          400
Name: count, dtype: int64

In [16]:
class NSERC_MD_Model(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [17]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.tolist())
X_test = embed(X_test_text.tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = NSERC_MD_Model(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [18]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 1055.6442
Epoch 2: Loss = 809.1634
Epoch 3: Loss = 766.6433
Epoch 4: Loss = 748.1599
Epoch 5: Loss = 734.6144
Epoch 6: Loss = 726.0840
Epoch 7: Loss = 717.0983
Epoch 8: Loss = 710.1387
Epoch 9: Loss = 703.4077
Epoch 10: Loss = 697.5927
Epoch 11: Loss = 692.1368
Epoch 12: Loss = 687.6844
Epoch 13: Loss = 682.0415
Epoch 14: Loss = 677.0149
Epoch 15: Loss = 673.0825
Epoch 16: Loss = 668.0252
Epoch 17: Loss = 663.9300
Epoch 18: Loss = 657.2137
Epoch 19: Loss = 653.9226
Epoch 20: Loss = 650.2271
Epoch 21: Loss = 647.2431
Epoch 22: Loss = 643.1177
Epoch 23: Loss = 638.5887
Epoch 24: Loss = 634.9138
Epoch 25: Loss = 630.7080
Epoch 26: Loss = 626.4862
Epoch 27: Loss = 623.6128
Epoch 28: Loss = 620.3366
Epoch 29: Loss = 614.4671
Epoch 30: Loss = 612.6786
Epoch 31: Loss = 608.2702
Epoch 32: Loss = 604.2969
Epoch 33: Loss = 602.0961
Epoch 34: Loss = 597.5246
Epoch 35: Loss = 594.8192
Epoch 36: Loss = 590.6971
Epoch 37: Loss = 585.9099
Epoch 38: Loss = 584.6702
Epoch 39: Loss = 578

In [19]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()

print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                         precision    recall  f1-score   support

               Advancement of knowledge       0.82      0.91      0.86      4036
Agriculture and primary food production       0.82      0.77      0.79       308
                    Commercial services       0.72      0.62      0.67        80
 Construction, urban and rural planning       0.80      0.73      0.77       275
                       Energy resources       0.80      0.80      0.80       625
                            Environment       0.79      0.74      0.76       786
  Health, education and social services       0.80      0.73      0.77       588
 Information and communication services       0.86      0.77      0.81       787
   Manufacturing processes and products       0.69      0.70      0.70       925
   Natural resources (economic aspects)       0.80      0.69      0.74       221
                   Northern development       0.79      0.64      0.71       241
  The socioeconomic objecti

In [20]:
torch.save(clf_model.state_dict(), "models/NSERC_MD.pt")

Model for Area of Research

In [21]:
# translator = GoogleTranslator(target="en")

# tmp_data = NSERC_DATA.sample(frac=1).reset_index(drop=True) # shuffle


# val_counts = tmp_data["Area_of_Research"].value_counts()
# classes = val_counts[val_counts > 50].index
# tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

# for class_name in classes:
#     tmp_data["Area_of_Research"] = tmp_data["Area_of_Research"].replace(class_name, translator.translate(class_name))

# val_counts = tmp_data["Area_of_Research"].value_counts()
# classes = val_counts[val_counts > 100].index
# tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

# tmp_data['Area_of_Research'] = tmp_data['Area_of_Research'].str.replace(r' \(.*?\)', '', regex=True)

# tmp_data = tmp_data.groupby(["Area_of_Research"]).head(500)
# tmp_data = tmp_data[tmp_data["Area_of_Research"] != "Not available"]

# tmp_data["Area_of_Research"].value_counts()


In [22]:
# class NSERC_AR_Model(nn.Module):
#     def __init__(self, input_dim, output_dim):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(input_dim, 256),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, output_dim)
#         )

#     def forward(self, x):
#         return self.net(x)

In [23]:
# X_data = embed(tmp_data['Title'].tolist())
# y_data = tmp_data['Area_of_Research'].astype('category').cat.codes
# label_mapping = dict(enumerate(tmp_data['Area_of_Research'].astype('category').cat.categories))

# X_train_text, X_test_text, y_train, y_test = train_test_split(
#     tmp_data['Title'], tmp_data['Area_of_Research'], test_size=0.2, stratify=tmp_data['Area_of_Research'], #random_state=42
# )

# X_train = embed(X_train_text)
# X_test = embed(X_test_text)
# y_train = torch.tensor(y_train.values)
# y_test = torch.tensor(y_test.values)

# input_dim = X_train.shape[1]
# output_dim = len(label_mapping)

# clf_model = NSERC_AR_Model(input_dim, output_dim).to(device)
# loss_fn = nn.CrossEntropyLoss()
# optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

# train_dataset = TensorDataset(X_train, y_train)
# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [24]:
# # Train
# for epoch in range(5):
#     clf_model.train()
#     total_loss = 0
#     for xb, yb in train_loader:
#         xb, yb = xb.to(device), yb.to(device)
#         optimizer.zero_grad()
#         preds = clf_model(xb)
#         loss = loss_fn(preds, yb)
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()
#     print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

In [25]:
# clf_model.eval()
# with torch.no_grad():
#     preds = clf_model(X_test.to(device))
#     pred_labels = preds.argmax(dim=1).cpu().numpy()
#     true_labels = y_test.numpy()

# print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values())))

In [26]:
# torch.save(clf_model.state_dict(), "models/NSERC_AR.pt")

SSHRC

In [27]:
sshrc_path = "raw_data/SSHRC/"
sshrc_files = Path(sshrc_path).glob("*.csv")

SSHRC_DFS = [pd.read_csv(f) for f in sshrc_files]
SSHRC_DATA = pd.concat(SSHRC_DFS, ignore_index=True)

In [28]:
grant_descriptors = [
    "Title-Titre", "Main_Discipline", "Area_of_Research", "Keywords-Mots-clés"
]

col_names = [
    'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]


SSHRC_DATA = SSHRC_DATA[grant_descriptors]
SSHRC_DATA.columns = col_names

SSHRC_DATA.drop_duplicates(inplace=True)

In [ ]:
SSHRC_DATA["Main_Discipline"].value_counts()

tmp_data = SSHRC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

val_counts = tmp_data["Main_Discipline"].value_counts()
classes = val_counts[val_counts > 200].index
tmp_data = tmp_data[tmp_data["Main_Discipline"].isin(classes)]

tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data.dropna(subset=['Main_Discipline'])
tmp_data = tmp_data[~(tmp_data['Main_Discipline'].isin(["Not Specified", "Not Applicable", "Multiple primary fields of research", "Interdisciplinary Studies"]))]


tmp_data["Main_Discipline"].value_counts()


Main_Discipline
Anthropology                                         500
Geography                                            500
Linguistics                                          500
History                                              500
Communications and Media Studies                     500
Literature, Modern Languages and                     500
Law                                                  500
Urban and Regional Studies, Environmental Studies    500
Fine Arts                                            500
Social Work                                          500
Philosophy                                           500
Economics                                            500
Sociology                                            500
Political Science                                    500
Management, Business, Administrative Studies         500
Education                                            500
Psychology                                           500
Not Specified  

In [30]:
class SSHRC_MD_Model(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [39]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = SSHRC_MD_Model(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [40]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 460.5351
Epoch 2: Loss = 436.9632
Epoch 3: Loss = 408.4201
Epoch 4: Loss = 377.2047
Epoch 5: Loss = 350.7558
Epoch 6: Loss = 330.2530
Epoch 7: Loss = 317.0723
Epoch 8: Loss = 305.0127
Epoch 9: Loss = 297.2562
Epoch 10: Loss = 290.6647
Epoch 11: Loss = 285.1106
Epoch 12: Loss = 281.1621
Epoch 13: Loss = 276.3270
Epoch 14: Loss = 273.5069
Epoch 15: Loss = 270.1561
Epoch 16: Loss = 267.6716
Epoch 17: Loss = 265.1463
Epoch 18: Loss = 262.6920
Epoch 19: Loss = 260.3654
Epoch 20: Loss = 258.3082
Epoch 21: Loss = 256.6513
Epoch 22: Loss = 254.4720
Epoch 23: Loss = 253.9725
Epoch 24: Loss = 251.8271
Epoch 25: Loss = 249.2476
Epoch 26: Loss = 248.5374
Epoch 27: Loss = 246.7361
Epoch 28: Loss = 245.4785
Epoch 29: Loss = 244.3219
Epoch 30: Loss = 242.9111
Epoch 31: Loss = 241.9365
Epoch 32: Loss = 241.3078
Epoch 33: Loss = 238.9769
Epoch 34: Loss = 237.7174
Epoch 35: Loss = 237.2085
Epoch 36: Loss = 236.0249
Epoch 37: Loss = 235.9558
Epoch 38: Loss = 234.0914
Epoch 39: Loss = 232.

In [41]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()

print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values())))

                                                   precision    recall  f1-score   support

                                     Anthropology       0.37      0.37      0.37       100
                                      Archaeology       0.34      0.41      0.37        81
                                       Archeology       0.00      0.00      0.00        31
             Classics, Classical & Dead Languages       0.58      0.56      0.57        39
                 Communications and Media Studies       0.39      0.36      0.37       100
                                      Criminology       0.57      0.62      0.59        92
                                       Demography       0.27      0.24      0.25        38
                                        Economics       0.53      0.46      0.49       100
                                        Education       0.41      0.38      0.39       100
                                        Fine Arts       0.46      0.45      0.45       10

In [42]:
torch.save(clf_model.state_dict(), "models/SSHRC_MD.pt")

Model for Area of Research

In [ ]:
# SSHRC_DATA["Area_of_Research"].value_counts()

# tmp_data = SSHRC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

# val_counts = tmp_data["Area_of_Research"].value_counts()
# classes = val_counts[val_counts > 100].index
# tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

# tmp_data = tmp_data.groupby(["Area_of_Research"]).head(500)
# tmp_data = tmp_data[~(tmp_data['Area_of_Research'].isin(["Not Subject to Research Classification", "Not Specified"]))]

# tmp_data["Area_of_Research"].value_counts()


In [ ]:
# class SSHRC_AR_Model(nn.Module):
#     def __init__(self, input_dim, output_dim):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(input_dim, 256),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, output_dim)
#         )

#     def forward(self, x):
#         return self.net(x)

In [ ]:
# X_data = embed(tmp_data['Title'].tolist())
# y_data = tmp_data['Area_of_Research'].astype('category').cat.codes
# label_mapping = dict(enumerate(tmp_data['Area_of_Research'].astype('category').cat.categories))

# X_train_text, X_test_text, y_train, y_test = train_test_split(
#     tmp_data['Title'], tmp_data['Area_of_Research'], test_size=0.2, stratify=tmp_data['Area_of_Research'], #random_state=42
# )

# X_train = embed(X_train_text)
# X_test = embed(X_test_text)
# y_train = torch.tensor(y_train.values)
# y_test = torch.tensor(y_test.values)

# input_dim = X_train.shape[1]
# output_dim = len(label_mapping)

# clf_model = SSHRC_AR_Model(input_dim, output_dim).to(device)
# loss_fn = nn.CrossEntropyLoss()
# optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

# train_dataset = TensorDataset(X_train, y_train)
# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [ ]:
# clf_model.eval()
# with torch.no_grad():
#     preds = clf_model(X_test.to(device))
#     pred_labels = preds.argmax(dim=1).cpu().numpy()
#     true_labels = y_test.numpy()

# print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values())))

In [ ]:
# torch.save(clf_model.state_dict(), "models/SSHRC_AR.pt")

In [ ]:
# embedder.save("models/embedder")